## Pauli Propagation: Classically estimating expectation values from parameterized quantum circuits.

In [ ]:
import pennylane as qp
import numpy as np
from itertools import combinations, product

In [ ]:
# action of cnot on pauli-op

cnot = qp.CNOT([0, 1])

for op0, op1 in product([qp.Identity, qp.X, qp.Y, qp.Z], repeat=2):
    original_op = op0(0) @ op1(1)
    new_op = cnot @ original_op @ cnot
    new_op = qp.pauli_decompose(new_op.matrix())
    print(f"CNOT transformed {original_op} to {new_op}")

In [ ]:
cnot_table = {
    ("I", "I"): (("I", "I"), 1),
    ("I", "X"): (("I", "X"), 1),
    ("I", "Y"): (("Z", "Y"), 1),
    ("I", "Z"): (("Z", "Z"), 1),
    ("X", "I"): (("X", "X"), 1),
    ("X", "X"): (("X", "I"), 1),
    ("X", "Y"): (("Y", "Z"), 1),
    ("X", "Z"): (("Y", "Y"), -1),
    ("Y", "I"): (("Y", "X"), 1),
    ("Y", "X"): (("Y", "I"), 1),
    ("Y", "Y"): (("X", "Z"), -1),
    ("Y", "Z"): (("X", "Y"), 1),
    ("Z", "I"): (("Z", "I"), 1),
    ("Z", "X"): (("Z", "X"), 1),
    ("Z", "Y"): (("I", "Y"), 1),
    ("Z", "Z"): (("I", "Z"), 1),
}

cnot_table

## Truncating Pauli-Propagation

In [ ]:
from pennylane.pauli import PauliWord, PauliSentence

In [ ]:
num_qubits = 4


H_coeffs = np.random.random((num_qubits - 1) * 3)
H_ops = [op(j) @ op(j + 1) for j in range(num_qubits - 1) for op in [qp.X, qp.Y, qp.Z]]
H = qp.dot(H_coeffs, H_ops)


H

In [ ]:
H.pauli_rep

In [ ]:
def _ansatz(params, num_qubits, H):
    """Parametrized quantum circuit ansatz that alternates arbitrary single-qubit
    rotations with strongly entangling CNOT layers. The depth of the ansatz and the
    number of qubits are given by the first dimension of the input parameters."""

    for i, params_layer in enumerate(params):
        # Execute arbitrary parametrized single-qubit rotations
        for j, params_qubit in enumerate(params_layer):
            qp.RZ(params_qubit[0], j)
            qp.RY(params_qubit[1], j)
            qp.RZ(params_qubit[2], j)
        # If we are not in the last layer, execute an entangling CNOT layer
        if i < len(params) - 1:
            for j in range(num_qubits):
                qp.CNOT([j, (j + 1) % num_qubits])

    return qp.expval(H)


ansatz = qp.transforms.make_tape(_ansatz)

In [ ]:
num_qubits = 4
num_layers = 3
np.random.seed(852)

params = np.random.random((num_layers, num_qubits, 3))
tape = ansatz(params, num_qubits, H)
print(qp.drawer.tape_text(tape))

In [ ]:
for op in reversed(tape.operations):
    print(op,op.wires)

In [ ]:
H = tape.measurements[0].obs.pauli_rep
H

In [ ]:
k = 8

for op in reversed(tape.operations):
    
    # propagating it across CNOT
    if isinstance(op, qp.CNOT):

        new_H = PauliSentence()
        for pauli_word, coeff in H.items():
            print('The observable to propageted:',pauli_word)
            # print(op)
            op_pw_0 = pauli_word.get(op.wires[0], "I")
            op_pw_1 = pauli_word.get(op.wires[1], "I")
            print(f"Form of observable before propagting through CX on {op.wires} is:",op_pw_0, op_pw_1)
            (new_op_pw_0, new_op_pw_1), factor = cnot_table[(op_pw_0, op_pw_1)]
            print(f'Form of the observable after propagating through CX on {op.wires} is:', factor, new_op_pw_0, new_op_pw_1)

            new_pw = pauli_word.copy()
            new_pw.update({op.wires[0]: new_op_pw_0, op.wires[1]: new_op_pw_1})
            new_pw = PauliWord(new_pw)
            print(f'New Pauli Word :', new_pw)

            if (k is None) or len(new_pw) <= k:
                new_H[new_pw] += factor*coeff
        
                print(new_H)

            print('\n------------------------\n')
        
        H = new_H
        print('\n------------------------\n')



In [ ]:
k = 8

H = tape.measurements[0].obs.pauli_rep

for op in reversed(tape.operations):
    
    if isinstance(op, (qp.RZ, qp.RX, qp.RY)):
        
        print(op)
        pauli = op.name[-1]
        wire = op.wires[0]
        params = op.wires[0]
        new_H = PauliSentence()

        rot_pauli_word = PauliWord({wire: pauli})
        print(rot_pauli_word)
        
        for pauli_word, coeff in H.items():
            print(pauli_word)
            
            if pauli_word.commutes_with(rot_pauli_word):
                # Rotation generator commutes with Pauli word from H, the word is unchanged
                new_H[pauli_word] += coeff
                # print(new_H)

            else:
                # Rotation generator does not commute with Pauli word from H;
                # multiply old coefficient by cosine, and add new term with modified Pauli word

                new_H[pauli_word] += qp.math.cos(params)*coeff
                new_pauli_word, factor = list((rot_pauli_word @ pauli_word).items())[0]
                new_H[new_pauli_word] += (qp.math.sin(params) * coeff * factor * 1j).real

            
            print(new_H)

        
        print('---------------\n')
        




    

In [ ]:
k = 8

H = tape.measurements[0].obs.pauli_rep

print('\n------------------------\n')
print(H)
print('\n------------------------\n')

for op in reversed(tape.operations):
    
    # propagating it across CNOT
    print(op)
    if isinstance(op, qp.CNOT):

        new_H = PauliSentence()
        for pauli_word, coeff in H.items():
            # print('The observable to propageted:',pauli_word)
            # print(op)
            op_pw_0 = pauli_word.get(op.wires[0], "I")
            op_pw_1 = pauli_word.get(op.wires[1], "I")
            # print(f"Form of observable before propagting through CX on {op.wires} is:",op_pw_0, op_pw_1)
            (new_op_pw_0, new_op_pw_1), factor = cnot_table[(op_pw_0, op_pw_1)]
            # print(f'Form of the observable after propagating through CX on {op.wires} is:', factor, new_op_pw_0, new_op_pw_1)

            new_pw = pauli_word.copy()
            new_pw.update({op.wires[0]: new_op_pw_0, op.wires[1]: new_op_pw_1})
            new_pw = PauliWord(new_pw)
            # print(f'New Pauli Word :', new_pw)

            if (k is None) or len(new_pw) <= k:
                new_H[new_pw] += factor*coeff
        
                # print(new_H)

        #     # print('\n------------------------\n')
        H = new_H
        # # print('\n------------------------\n')
    
    elif isinstance(op, (qp.RZ, qp.RX, qp.RY)):
        pauli = op.name[-1]
        wire = op.wires[0]
        params = op.wires[0]
        new_H = PauliSentence()

        rot_pauli_word = PauliWord({wire: pauli})
        # print(rot_pauli_word)
        
        for pauli_word, coeff in H.items():
            # print(pauli_word)
            
            if pauli_word.commutes_with(rot_pauli_word):
                # Rotation generator commutes with Pauli word from H, the word is unchanged
                new_H[pauli_word] += coeff
                # print(new_H)

            else:
                # Rotation generator does not commute with Pauli word from H;
                # multiply old coefficient by cosine, and add new term with modified Pauli word

                new_H[pauli_word] += qp.math.cos(params)*coeff
                new_pauli_word, factor = list((rot_pauli_word @ pauli_word).items())[0]
                new_H[new_pauli_word] += (qp.math.sin(params) * coeff * factor * 1j).real

            
            # print(new_H)

            H = new_H

        
    print('\n------------------------\n')
    print('new Hamiltoninan', H)
    print('\n------------------------\n')



# Function Form

In [ ]:
def _ansatz(params, num_qubits, H):
    """Parametrized quantum circuit ansatz that alternates arbitrary single-qubit
    rotations with strongly entangling CNOT layers. The depth of the ansatz and the
    number of qubits are given by the first dimension of the input parameters."""

    for i, params_layer in enumerate(params):
        # Execute arbitrary parametrized single-qubit rotations
        for j, params_qubit in enumerate(params_layer):
            qp.RZ(params_qubit[0], j)
            qp.RY(params_qubit[1], j)
            qp.RZ(params_qubit[2], j)
        # If we are not in the last layer, execute an entangling CNOT layer
        if i < len(params) - 1:
            for j in range(num_qubits):
                qp.CNOT([j, (j + 1) % num_qubits])

    return qp.expval(H)


ansatz = qp.transforms.make_tape(_ansatz)

In [ ]:
def apply_cnot(wires, H, k=None):
    """Apply a CNOT gate on given wires to operator H in the Heisenberg picture.
    Truncate all Pauli words in the transformed operator that have weight larger than k."""
    new_H = PauliSentence()
    for pauli_word, coeff in H.items():
        # Extract the Pauli tensor factors on the wires of the CNOT
        op_pw_0 = pauli_word.get(wires[0], "I")
        op_pw_1 = pauli_word.get(wires[1], "I")
        # Look up the prefactor and new Pauli tensor factors in our lookup table
        (new_op_pw_0, new_op_pw_1), factor = cnot_table[(op_pw_0, op_pw_1)]
        # Create new Pauli word from old one and update it with new tensor factors
        new_pw = pauli_word.copy()
        new_pw.update({wires[0]: new_op_pw_0, wires[1]: new_op_pw_1})
        new_pw = PauliWord(new_pw)

        # Truncation: Only add to the new H if the new Pauli word is small enough
        if (k is None) or len(new_pw) <= k:
            new_H[new_pw] += factor * coeff

    return new_H


def apply_single_qubit_rot(pauli, wire, param, H):
    """Apply a single-qubit rotation about the given ``pauli`` on the given ``wire``
    by a rotation angle ``param`` to an operator ``H``."""
    new_H = PauliSentence()
    rot_pauli_word = PauliWord({wire: pauli})
    for pauli_word, coeff in H.items():
        if pauli_word.commutes_with(rot_pauli_word):
            # Rotation generator commutes with Pauli word from H, the word is unchanged
            new_H[pauli_word] += coeff
        else:
            # Rotation generator does not commute with Pauli word from H;
            # Multiply old coefficient by cosine, and add new term with modified Pauli word
            new_H[pauli_word] += qp.math.cos(param) * coeff
            new_pauli_word, factor = list((rot_pauli_word @ pauli_word).items())[0]
            new_H[new_pauli_word] += (qp.math.sin(param) * coeff * factor * 1j).real

    return new_H


In [ ]:
def initial_state_expval(H):
    """Compute the expectation value of an operator ``H`` in the state |0>."""
    expval = 0.0
    for pauli_word, coeff in H.items():
        if all(pauli in {"I", "Z"} for pauli in pauli_word.values()):
            expval += coeff
    return expval

In [ ]:
def execute_tape(tape, k=None):
    """Classically simulate a tape and estimate the expectation value
    of its output observable using truncated Pauli propagation."""
    H = tape.measurements[0].obs.pauli_rep
    for op in reversed(tape.operations):
        if isinstance(op, qp.CNOT):
            # Apply CNOT
            H = apply_cnot(op.wires, H, k=k)
        elif isinstance(op, (qp.RZ, qp.RX, qp.RY)):
            # Extract the Pauli rotation generator, wire, and parameter from the gate
            pauli = op.name[-1]
            wire = op.wires[0]
            param = op.data[0]
            H = apply_single_qubit_rot(pauli, wire, param, H)
        else:
            raise NotImplementedError

    return initial_state_expval(H)

In [ ]:
num_qubits = 25
num_layers = 5
k = 9
H_coeffs = np.random.random((num_qubits - 1) * 3)
H_ops = [op(j) @ op(j + 1) for j in range(num_qubits - 1) for op in [qp.X, qp.Y, qp.Z]]
H = qp.dot(H_coeffs, H_ops)
print(H)
params = np.random.random((num_layers, num_qubits, 3))


def run_estimate(params, H):
    tape = ansatz(params, num_qubits, H)
    expval = execute_tape(tape, k=k)
    return expval


expval = run_estimate(params, H)


@qp.qnode(qp.device("lightning.qubit", wires=num_qubits))
def run_lightning(params, H):
    return _ansatz(params, num_qubits, H)


exact_expval = run_lightning(params, H)

print(f"Expectation value estimated by truncated Pauli propagation: {expval:.6f}")
print(f"Numerically exact expectation value:                        {exact_expval:.6f}")